# Notebook 5 of 7: The Transformer Revolution

## The Paper That Changed Everything

**Series: Understanding AI Through Italian Music**

---

In 2017, a team of researchers at Google published a paper with a simple but bold title: **"Attention Is All You Need."**

That paper introduced the **Transformer** -- a new way of building AI models that would go on to become the foundation of virtually every major AI system you have heard of:

- **ChatGPT** (OpenAI)
- **Gemini** (Google)
- **Claude** (Anthropic)
- **Copilot** (Microsoft)

All of them are built on the Transformer architecture.

In the previous notebooks, we trained an **RNN** (Notebook 3) and an **LSTM** (Notebook 4) on Italian song lyrics. Both could generate text, but both had limitations -- especially with remembering context and producing coherent writing.

In this notebook, we will use **GPT-2**, a Transformer model released by OpenAI in 2019, and see firsthand why it produces dramatically better results. GPT-2 is the great-grandparent of ChatGPT, and it uses the exact same architecture -- just at a smaller scale.

In this notebook, we will:

1. **Understand** the three key ideas that make Transformers so powerful
2. **Load** a pre-trained GPT-2 model and see its massive scale
3. **Fine-tune** it on Italian lyrics (teaching an English-speaking AI to write Italian songs)
4. **Compare** its output to the RNN and LSTM from earlier notebooks
5. **Connect** this small experiment to the real-world AI systems you use every day

Let's meet the architecture that changed everything.

## What Makes Transformers Different?

The RNN and LSTM we built in earlier notebooks read text **one word at a time**, left to right, like a person reading aloud. The Transformer does something fundamentally different. Three key ideas set it apart:

### 1. Attention: Seeing Everything at Once

Instead of reading one word at a time, the Transformer looks at **all the words simultaneously** and figures out which words relate to each other.

Consider this sentence:

> *"The cat sat on the mat because **it** was tired."*

What does "it" refer to? You instantly know it means "the cat" -- not "the mat." But how? You looked back across the whole sentence and made a connection. That is **attention**.

An RNN would struggle here because by the time it reaches "it," the memory of "the cat" from the beginning of the sentence has already faded. The Transformer sees the whole sentence at once, so it can directly connect "it" to "the cat" no matter how far apart they are.

### 2. Transfer Learning: Standing on the Shoulders of Giants

Our RNN and LSTM started from **complete ignorance** -- random numbers that knew nothing about language. They had to learn everything from scratch using only our 500 Italian songs.

GPT-2 is different. It has already read **millions of web pages** in English. It already knows about grammar, sentence structure, common phrases, and even some facts about the world. When we train it on Italian lyrics, we are not starting from zero -- we are **fine-tuning** an AI that already understands language.

Think of it like this: teaching someone who already knows music theory to play a new genre is much easier than teaching someone who has never heard music before.

### 3. Scale: 124 Million Reasons

Our RNN had about 13 million parameters. Our LSTM had about 26 million. GPT-2 has **124 million parameters** -- roughly 5 to 10 times larger.

More parameters means more capacity to learn subtle patterns. It is the difference between sketching with a pencil (few parameters) and painting with a full set of oils (many parameters). Both can create art, but one can capture far more nuance and detail.

---

These three ideas -- attention, transfer learning, and scale -- are what make modern AI so powerful. Let's see them in action.

## Setup

Run the cell below to load all the tools we need. This time, we are importing the **Transformer-specific** functions: `create_transformer_model` for loading GPT-2, and `generate_transformer` for generating text with it.

You do not need to understand every import. Just run the cell and move on.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Our custom modules
from src.dataset import ItalianLyricsDataset, load_lyrics
from src.models import RNNModel, LSTMModel, create_transformer_model, model_summary
from src.training import train_model, get_optimizer
from src.generation import generate_transformer
from src.visualization import plot_training_loss, plot_parameter_comparison, display_generation_comparison

# Show charts inside the notebook
%matplotlib inline

# Detect if a GPU is available (training is faster on GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("All tools loaded successfully!")

## Loading a Pre-Trained Model

This is the key moment where the Transformer experience diverges from what we did with RNN and LSTM.

When we built the RNN and LSTM, we created them **from scratch** -- empty models filled with random numbers. They knew absolutely nothing about language, words, or Italian.

GPT-2 is different. When we load it, we are downloading a model that **already knows English**. Those 124 million parameters encode knowledge about language structure, grammar, vocabulary, and even some facts about the world -- all learned from reading roughly **8 million web pages**.

Now we will **fine-tune** it -- adjust those parameters slightly so it learns the patterns of Italian song lyrics. We are not teaching it language from scratch. We are teaching an English speaker to write Italian songs.

In [ ]:
# Load the GPT-2 tokenizer (same one used in Notebooks 2-4)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Load the pre-trained GPT-2 model
# This downloads ~500MB of pre-trained knowledge
transformer_model = create_transformer_model(tokenizer)
transformer_model = transformer_model.to(device)

# Show the model's size
print("GPT-2 Transformer:")
print("-" * 40)
model_summary(transformer_model, "GPT-2 Transformer")

## Compare Model Sizes

Numbers are hard to feel. Let's make the scale difference **visual**.

Below, we create all three models -- RNN, LSTM, and GPT-2 Transformer -- and plot their parameter counts side by side. Remember: parameters are the individual numbers inside the model that get adjusted during training. More parameters means more capacity to learn patterns.

When you see the chart, notice how the Transformer absolutely dwarfs the other two. This is why it can capture much more nuanced patterns in language.

In [ ]:
vocab_size = len(tokenizer)

# Create all three models for comparison
rnn_model = RNNModel(vocab_size, embedding_dim=256, hidden_dim=512)
lstm_model = LSTMModel(vocab_size, embedding_dim=256, hidden_dim=512)

# Count parameters for each
rnn_params = model_summary(rnn_model, "RNN")
lstm_params = model_summary(lstm_model, "LSTM")
transformer_params = model_summary(transformer_model, "GPT-2 Transformer")

print()
print(f"The Transformer has {transformer_params / rnn_params:.0f}x more parameters than the RNN")
print(f"The Transformer has {transformer_params / lstm_params:.0f}x more parameters than the LSTM")

# Visualize the difference
param_counts = {
    "RNN": rnn_params,
    "LSTM": lstm_params,
    "Transformer\n(GPT-2)": transformer_params
}
fig = plot_parameter_comparison(param_counts)

## Prepare the Data and Train

Now comes the exciting part: **fine-tuning** GPT-2 on Italian song lyrics.

We will load 500 Italian songs and train for just **2 epochs** (two complete passes through all the data). Compare this to the 5 epochs we used for the RNN and LSTM in earlier notebooks.

Why fewer epochs? Two reasons:

1. **GPT-2 is much slower to train.** With 124 million parameters to adjust (vs. ~13 million for RNN), each training step takes much longer. This is the real-world cost of scale.

2. **GPT-2 already knows a lot.** Because it was pre-trained on millions of web pages, it does not need to learn language from scratch. A few passes through our data is enough to adapt its existing knowledge to Italian lyrics. Think of it as a quick lesson, not a full course.

**Note:** This cell will take several minutes to run. That is normal -- you are witnessing firsthand why training large AI models is expensive.

In [ ]:
import time

# Load 500 Italian songs
lyrics = load_lyrics('../data/italian_lyrics.txt', max_songs=500)
print(f"Loaded {len(lyrics)} Italian songs")

# Create the dataset and dataloader
dataset = ItalianLyricsDataset(lyrics, tokenizer, max_length=128)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Use a very small learning rate for the Transformer
# (we don't want to destroy its pre-trained knowledge)
optimizer = get_optimizer(transformer_model, is_transformer=True)

# Train for 2 epochs
print("\nTraining GPT-2 on Italian lyrics...")
print("(This will take several minutes -- the price of power)\n")

start = time.time()
transformer_history = train_model(
    transformer_model, dataloader, optimizer, device,
    epochs=2, is_transformer=True, model_name="Transformer"
)
elapsed = time.time() - start
print(f"\nTotal training time: {elapsed / 60:.1f} minutes")

## Generate Italian Lyrics

The moment of truth. Let's give our fine-tuned Transformer the seed phrase **"Amore mio"** ("My love") and see what it writes.

Then we will compare its output to what our RNN and LSTM typically produced in earlier notebooks. If you ran those notebooks, you may remember the results: lots of repetition, fragments that trailed off, and text that rarely made grammatical sense.

Let's see if the Transformer does better.

In [ ]:
# Generate lyrics with the Transformer
seed_text = "Amore mio"

print(f'Seed text: "{seed_text}"\n')
print("=" * 60)
print("  TRANSFORMER (GPT-2) OUTPUT")
print("=" * 60)

transformer_output = generate_transformer(
    transformer_model, tokenizer, seed_text,
    max_length=100, temperature=0.7, top_k=50, top_p=0.9
)
print(transformer_output)

print("\n" + "=" * 60)
print("  WHAT RNN/LSTM TYPICALLY PRODUCE (from earlier notebooks)")
print("=" * 60)
print("""
RNN:  "Amore mio che che che che il il sole sole sole
       sole sole che non non non non..."

LSTM: "Amore mio non so che cosa fare il mio cuore
       il mio cuore il mio cuore il mio cuore..."

Notice the difference? The RNN gets stuck in loops almost
immediately. The LSTM holds on a bit longer but still
falls into repetition. The Transformer produces text that
flows more naturally and stays coherent longer.
""")

## Why Is It Better?

You just saw the Transformer produce noticeably better text than the RNN or LSTM. Here is why, broken down into the three advantages we discussed at the beginning:

### 1. Attention Sees the Whole Context

The RNN reads one word at a time and tries to remember what came before. By the time it has generated 20 words, it has largely forgotten the beginning. This is why it loops -- it loses track of what it has already said.

The Transformer uses **attention** to look at the **entire sequence** at every step. When deciding the 50th word, it can look all the way back to the 1st word. This is why its output stays more coherent and avoids falling into repetitive cycles.

### 2. Pre-Training Gave It a Foundation

Our RNN and LSTM had to learn *everything* from 500 Italian songs -- grammar, vocabulary, sentence structure, how words relate to each other. That is like trying to learn a language by reading 500 texts with no dictionary or teacher.

GPT-2 already understood language structure from reading millions of English web pages. When we fine-tuned it, it only had to learn the **style and vocabulary** of Italian lyrics. The heavy lifting was already done.

### 3. More Parameters = More Nuance

With 124 million parameters (vs. ~13 million for RNN), the Transformer has roughly 10 times more "brain cells" available to encode patterns. It can learn subtle things like how the mood of a song shifts between verses, or how certain Italian word combinations sound poetic together.

### The Trade-Off

But this power comes at a cost. The Transformer took **much longer to train**, even for just 2 epochs (vs. 5 epochs for the simpler models). In the real world, this trade-off between quality and cost is one of the most important decisions in AI development.

## The Cost of Power

Let's put real numbers on the trade-off we just discussed.

Below, we compare how long each model type takes to train. Even in our tiny experiment (500 songs, a few epochs), the difference is dramatic. Now imagine scaling this up to the real world, where frontier AI models train on billions of documents for weeks or months.

If our small GPT-2 experiment takes several minutes for 2 epochs, imagine training GPT-4 with **10,000 times more parameters** on **1,000,000 times more data**. This is why training frontier AI models costs **tens of millions of dollars** and consumes enormous amounts of energy -- enough to power thousands of homes.

In [ ]:
from src.visualization import plot_training_time_comparison

# We'll estimate RNN and LSTM training times for comparison
# (In a full run, you'd train all three and collect actual times)

# Typical training times from earlier notebooks (5 epochs, 500 songs):
rnn_history = {
    'model_name': 'RNN',
    'epoch_losses': [7.5, 6.8, 6.2, 5.9, 5.7],
    'training_time': 30  # ~30 seconds for 5 epochs
}
lstm_history = {
    'model_name': 'LSTM',
    'epoch_losses': [7.2, 6.3, 5.6, 5.2, 4.9],
    'training_time': 45  # ~45 seconds for 5 epochs
}

# Compare all three
all_histories = [rnn_history, lstm_history, transformer_history]

print("Training Time Comparison:")
print("-" * 40)
for h in all_histories:
    minutes = h['training_time'] / 60
    epochs_trained = len(h['epoch_losses'])
    print(f"  {h['model_name']:15s}: {minutes:5.1f} minutes ({epochs_trained} epochs)")

transformer_time = transformer_history['training_time']
rnn_time = rnn_history['training_time']
print(f"\nThe Transformer took {transformer_time / rnn_time:.0f}x longer than the RNN")
print("...and it only trained for 2 epochs instead of 5!")

# Visualize the comparison
fig = plot_training_time_comparison(all_histories)

## From GPT-2 to ChatGPT: The Same Architecture, Scaled Up

The GPT-2 model we just used is from **2019**. It is a direct ancestor of the AI systems you may use today. Here is how the family tree looks:

| Model | Year | Parameters | Training Data | What It Could Do |
|-------|------|-----------|---------------|-----------------|
| **GPT-2** | 2019 | 124 million | ~8 million web pages | Generate coherent paragraphs. What we just used. |
| **GPT-3** | 2020 | 175 billion (1,400x bigger) | ~500 billion words | Write essays, answer questions, translate languages. |
| **GPT-4** | 2023 | ~1.8 trillion (estimated) | Undisclosed (massive) | Powers ChatGPT. Passes bar exams. Analyzes images. |

The remarkable thing? **The core architecture is the same.** GPT-4 uses the same Transformer design from that 2017 "Attention Is All You Need" paper. The differences are:

1. **Scale** -- More parameters and more training data. A LOT more.
2. **Training refinement** -- Techniques like **RLHF** (Reinforcement Learning from Human Feedback) teach the model to be helpful, harmless, and honest. This is what makes ChatGPT *conversational* rather than just a text generator.
3. **Engineering** -- Thousands of GPUs running for months, with sophisticated systems to distribute the work.

What you did in this notebook -- loading a pre-trained model, fine-tuning it on specific data, and generating text -- is conceptually **the exact same process** used to build ChatGPT, Claude, and Gemini. The difference is scale: millions of dollars, months of training, and data from across the entire internet.

You have just walked the same path that led to the AI revolution. You did it with 500 Italian songs instead of 500 billion words, but the ideas are identical.

## Try Different Prompts

Now it is your turn to experiment. Change the `seed_text` below to any Italian phrase you like and see what the model writes. You can also adjust the generation settings:

- **temperature** (0.1 to 1.5): Controls randomness. Lower = more predictable and repetitive. Higher = more creative but potentially nonsensical. Try 0.3 for conservative output or 1.0 for wilder creativity.
- **top_k** (1 to 100): Only consider the top K most likely next words. Lower = more focused. Higher = more variety.
- **top_p** (0.1 to 1.0): Only consider words until their combined probability reaches this threshold. 0.9 is a good default.

Some seed texts to try:
- `"La notte"` (The night)
- `"Nel silenzio"` (In the silence)
- `"Io cammino"` (I walk)
- `"Il cielo sopra"` (The sky above)
- `"Dimmi perche"` (Tell me why)

In [ ]:
# === EXPERIMENT HERE ===
# Change these values and re-run the cell to see different results!

seed_text = "La notte"       # Try different Italian phrases
temperature = 0.7            # 0.1 = conservative, 1.5 = wild
top_k = 50                   # Number of top words to consider
top_p = 0.9                  # Probability threshold

# Generate!
print(f'Seed: "{seed_text}"')
print(f"Settings: temperature={temperature}, top_k={top_k}, top_p={top_p}")
print("-" * 60)

output = generate_transformer(
    transformer_model, tokenizer, seed_text,
    max_length=100, temperature=temperature,
    top_k=top_k, top_p=top_p
)
print(output)

# Try running this cell multiple times with the SAME settings.
# You'll get different results each time because generation
# involves randomness. This is by design -- it's what makes
# AI creative rather than mechanical.

## Key Takeaways

Here is what we learned in this notebook:

1. **Transformers see all words at once.** The attention mechanism lets the model connect any word to any other word in the sequence, no matter how far apart they are. This is why Transformer output stays coherent over longer passages.

2. **Transfer learning reuses existing knowledge.** GPT-2 was pre-trained on millions of web pages before we ever touched it. Fine-tuning adapts that existing knowledge to a new task (Italian lyrics) rather than learning from scratch. This is dramatically more efficient.

3. **Scale drives capability.** GPT-2's 124 million parameters give it far more capacity to learn patterns than our RNN (13 million) or LSTM (26 million). The same principle applies all the way up to GPT-4 (estimated 1.8 trillion parameters).

4. **Training is expensive.** More power requires more time and compute. Our Transformer took much longer to train than the simpler models, and real-world AI training costs millions of dollars.

5. **The architecture is the same from GPT-2 to GPT-4.** The Transformer design from 2017 is still the foundation of every major AI system in 2024. The difference is scale and refinement, not fundamental architecture.

---

**The big picture:** You have now trained all three major architectures used in modern AI -- from the simplest (RNN) through the foundation of today's AI revolution (Transformer). You have seen firsthand how attention, pre-training, and scale combine to produce dramatically better results.

## What's Next?

The Transformer generates better text, but it is still not perfect. Sometimes it repeats itself. Sometimes the output does not quite make sense. Sometimes it produces something wonderful and then drifts off into nonsense.

In the **next notebook** (Notebook 6: Improving Generation), we will explore the techniques that make generation quality much better -- and understand why AI sometimes says the same thing over and over. We will learn about:

- **Temperature** -- how randomness controls creativity
- **Top-k and top-p sampling** -- smarter ways to pick the next word
- **Repetition penalties** -- how to stop the model from getting stuck in loops
- **The fundamental trade-off** between creativity and coherence

These are the same techniques used in ChatGPT, Claude, and every other AI chatbot to produce the fluent, varied responses you see in real-world applications.

---

*Notebook 5 of 7 -- The Transformer Revolution -- Complete.*